# DistilGPT2 Text Generation

**Model:** distilbert/distilgpt2 | **Size:** 82MB | **Product:** prod-zxor57muhdwkm

A distilled version of GPT-2 that retains 97% of the original model's language generation capabilities at 40% less parameters. Ideal for lightweight text generation, autocomplete, and creative writing assistance where a full GPT-2 would be over-engineered.

## Use Cases
- Text autocomplete and writing assistance
- Lightweight content generation for templated copy
- Prototyping generative AI features before scaling to larger models
- Code comment and documentation generation

In [ ]:
import boto3
import sagemaker
from sagemaker import ModelPackage

region = boto3.Session().region_name
role = sagemaker.get_execution_role()
sm_client = boto3.client('sagemaker', region_name=region)

print(f'Region: {region}')
print(f'Role: {role}')

In [ ]:
# Replace with your actual Model Package ARN from AWS Marketplace
model_package_arn = 'arn:aws:sagemaker:REGION:ACCOUNT:model-package/MODEL_PACKAGE_NAME'

# Validate ARN before deploying
if 'REGION' in model_package_arn or 'ACCOUNT' in model_package_arn or 'MODEL_PACKAGE_NAME' in model_package_arn:
    raise ValueError(
        'model_package_arn contains placeholder values. '
        'Subscribe to the model on AWS Marketplace and replace with the actual ARN.'
    )

endpoint_name = 'distilgpt2-text-generation'
instance_type = 'ml.m5.xlarge'

try:
    model = ModelPackage(
        role=role,
        model_package_arn=model_package_arn,
        sagemaker_session=sagemaker.Session()
    )
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=instance_type,
        endpoint_name=endpoint_name
    )
    print(f'Endpoint deployed: {endpoint_name}')
except Exception as e:
    print(f'Deployment failed: {e}')
    raise

## Step 2: Run Inference

Send a text prompt for open-ended text generation. Control output length with `max_new_tokens`.

In [ ]:
import json

runtime = boto3.client('sagemaker-runtime', region_name=region)

# Sample prompts for text generation
prompts = [
    'The future of cloud computing is',
    'Once upon a time in a small village, a young engineer discovered',
    'The most important factor in building reliable software systems is',
]

for prompt in prompts:
    payload = json.dumps({
        'inputs': prompt,
        'parameters': {'max_new_tokens': 60, 'do_sample': True, 'temperature': 0.8}
    })
    try:
        response = runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Body=payload
        )
        result_raw = response['Body'].read().decode('utf-8')
        try:
            result = json.loads(result_raw)
            print(f'Prompt: "{prompt}"')
            print(f'Generated: {result}\n')
        except json.JSONDecodeError:
            print(f'Raw response: {result_raw}')
    except Exception as e:
        print(f'Inference failed: {e}')
        raise

In [ ]:
# Cleanup - delete the endpoint to avoid ongoing charges
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f'Endpoint {endpoint_name} deleted.')
except Exception as e:
    print(f'Cleanup failed: {e}')